**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/catalystneuro/encore", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/goncalves-2013"], check=True)
    os.chdir("/content/encore/entries/goncalves-2013")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Goncalves et al. (2013) from CRCNS ssc-8

**Paper.** Goncalves JT, Anstey JE, Golshani P, Portera-Cailliau C (2013). Circuit level defects in
the developing neocortex of Fragile X mice. *Nature Neuroscience* 16:903-909,
[doi:10.1038/nn.3415](https://doi.org/10.1038/nn.3415). The claims file
(`claims/crcns-ssc-8/10.1038-nn.3415.yaml`) was written from the version of record in PubMed Central
(PMC3695061): main text and Online Methods, without the figure legends or the supplement.

**Data set.** CRCNS ssc-8, [doi:10.6080/K0X63K3X](https://doi.org/10.6080/K0X63K3X), read from the
public bucket `crcnsarchive` under `ssc-8/live/`. CRCNS data sets are not versioned, and the paper
cites no version (it predates the deposit of 2017); the only published state is the one in
`ssc-8/live/`, data descriptor version 0.7 (13 October 2017), so that is the state used. It is
recorded as `unversioned` and pinned by the MD5 sums of the data set's `checksums.md5`, which the
next cell checks against the bytes read:
`firingrates_and_distances.mat` ae1b48123540ecfd1bc9675c1eaa9a3d and
`Ca2+_imaging_TIFFs.tar.gz` 75bf460dbe90c625637d1a1af3a25ca3 (the archive is streamed once, and
its members are hashed one by one, as the archive's own MD5 needs the whole compressed stream).

**What the data set holds.** For each of 53 unanesthetized mice (WT and Fmr1 -/-, three ages), the
estimated firing rate of each neuron in each frame, the cell-to-cell distances in microns, and the
raw movie. The whole-cell recordings, the EEG, the isoflurane imaging and the 15.6 Hz imaging of
the paper are not in it.

| Claim | Tested here |
|---|---|
| group-sizes | tested |
| fig1d-distance-dependent-decorrelation | tested |
| fig1e-age-decrease | tested |
| genotype-pooled-ages | tested |
| pearson-genotype | tested |
| fig1ef-p14-16-difference | tested |
| p30-40-no-difference | tested |
| fig2b-peak-frequency | tested |
| fig2c-participation | tested |
| imaging-firing-rate-no-genotype-difference | tested |
| unanesthetized-p14-16-values | tested |
| wholecell-updown-firing, wholecell-state-modulation, fig3f-foa-time, ap-kinetics-similar, fig4-updown-properties-similar, fig4f-spikes-per-up-state, fig4g-up-state-firing-probability | blocked: no whole-cell data |
| fig5c-firing-vs-lh-p14-16, p30-40-firing-vs-lh, fig5d-correlation-vs-lh, fig5ef-slow-wave-epochs, fig5g-lh-power-similar | blocked: no EEG |
| suppl4a-firing-vs-correlation | blocked: the 3-min recordings of the EEG subset are not identified (shown as a variant on 3-min segments) |
| fast-imaging-control | blocked: no 15.6 Hz imaging |
| fig6-isoflurane-wt-correlation, fig6-isoflurane-ko-correlation, fig6-isoflurane-firing | blocked: no isoflurane imaging |

**Readings shared by the claims** (claims file, header). The trace is the deposited estimated firing
rate (reading T-a); reading T-b, the low-pass filtered dF/F, needs the cells to be segmented anew
from the raw movies, and the data set holds no contours, so it cannot be run here. The correlation
is the zero-lag normalised cross-correlation without mean subtraction (C-b); Spearman (C-a) is a
variant only, and Pearson is the paper's own variant (claim pearson-genotype). Each mouse's value
is the mean over its pairs.

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.formula.api import ols

import ssc8

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", "{:.4g}".format)
warnings.filterwarnings("ignore", category=RuntimeWarning)

started = time.time()
published = ssc8.checksums()
data, mat_md5, mat_keys = ssc8.load_mat()
print(f"data set {ssc8.DATASET_ID} ({ssc8.DATASET_VERSION}, doi:{ssc8.DATASET_DOI})")
print("firingrates_and_distances.mat MD5", mat_md5, "matches checksums.md5:",
      mat_md5 == published["firingrates_and_distances.mat"])
print("published MD5 of the TIFF archive:", published["Ca2+_imaging_TIFFs.tar.gz"])
t0 = time.time()
movies = ssc8.survey_movies()
print(f"streamed the TIFF archive ({len(movies)} members) in {time.time() - t0:.0f} s")

## Data check

Every check of the descriptor against the values is stated under its output.

In [ ]:
print("Variables in the MAT file:", mat_keys)
rows = []
for (g, a), recs in data.items():
    for i, r in enumerate(recs):
        x, d = r["rates"], r["dist"]
        iu = np.triu_indices_from(d, 1)
        rows.append({"genotype": g, "age": ssc8.AGE_LABEL[a], "index": i, "movie": f"Movie_{i + 1}.tif",
                     "frames": x.shape[0], "neurons": x.shape[1], "dist_shape_ok": d.shape == (x.shape[1],) * 2,
                     "minutes_at_3.9Hz": x.shape[0] / ssc8.FRAME_HZ / 60,
                     "rate_min": x.min(), "rate_max": x.max(), "mean_rate": x.mean(),
                     "frac_frames_nonzero": (x > 0).mean(), "min_nonzero": x[x > 0].min(),
                     "silent_cells": int((x.sum(0) == 0).sum()), "nan": int(np.isnan(x).sum()),
                     "dist_symmetric": bool(np.allclose(d, d.T)), "dist_diag_max": np.abs(np.diag(d)).max(),
                     "dist_min_offdiag": d[iu].min(), "dist_max": d[iu].max(), "pairs": len(iu[0]),
                     "pairs_lt100": int((d[iu] < 100).sum())})
info = pd.DataFrame(rows)
print("\nMice per group (recordings in the MAT file):")
print(info.groupby(["genotype", "age"], sort=False).size().to_string())
print("\nNeurons per recording: %d to %d; frames %d to %d (%.1f to %.1f min at 3.9 Hz; median %.1f)"
      % (info.neurons.min(), info.neurons.max(), info.frames.min(), info.frames.max(),
         info["minutes_at_3.9Hz"].min(), info["minutes_at_3.9Hz"].max(), info["minutes_at_3.9Hz"].median()))
print("Recordings of exactly 2160 frames:", int((info.frames == 2160).sum()), "of", len(info))
print("Rates: min %.3g, max %.3g; mean over recordings of the mean rate %.3f; smallest non-zero value %.3f to %.3f"
      % (info.rate_min.min(), info.rate_max.max(), info.mean_rate.mean(), info.min_nonzero.min(), info.min_nonzero.max()))
print("Fraction of non-zero frames per recording: %.3f to %.3f" % (info.frac_frames_nonzero.min(), info.frac_frames_nonzero.max()))
print("Silent cells (all zero):", info.silent_cells.sum(), "; NaN values:", info.nan.sum())
print("Distance matrices square and matching N:", bool(info.dist_shape_ok.all()), "; symmetric:", bool(info.dist_symmetric.all()),
      "; zero diagonal:", bool((info.dist_diag_max == 0).all()))
print("Off-diagonal distances: %.1f to %.1f um" % (info.dist_min_offdiag.min(), info.dist_max.max()))
print(info.to_string(index=False))

In [ ]:
# Unit signature of the estimated rates. If the rates were whole spike counts per frame, or counts
# times 3.9 Hz, the non-zero values would fall on multiples of 1 or of 3.9. Among the values of at
# least half a step, the fraction within 0.01 of a multiple is compared with what continuous values
# give (0.02 / step).
nz = np.concatenate([r["rates"][r["rates"] > 0] for recs in data.values() for r in recs])
print("Non-zero rate values: %d, of which distinct (to 1e-6): %d" % (len(nz), len(np.unique(np.round(nz, 6)))))
print("Percentiles 0, 1, 5, 50, 95, 99, 100:", np.round(np.percentile(nz, [0, 1, 5, 50, 95, 99, 100]), 3).tolist())
for base in [1.0, 3.9, 4.0]:
    x = nz[nz >= base / 2]
    near = np.mean(np.abs(x - base * np.round(x / base)) < 0.01)
    print(f"fraction within 0.01 of a multiple of {base}: {near:.4f} (continuous values: {0.02 / base:.4f})")
print("Fraction of non-zero values at or above 1: %.3f; at or above 3.9: %.3f" % ((nz >= 1).mean(), (nz >= 3.9).mean()))
h, e = np.histogram(nz, bins=[0, 0.5, 1, 2, 3, 4, 6, 10, 20, 70])
print("Histogram of non-zero values:", dict(zip([f"{a:g}-{b:g}" for a, b in zip(e[:-1], e[1:])], h.tolist())))

The descriptor says 6 to 13 animals per group, 40 to 149 neurons per animal, 3.9 Hz and about 7 to
8 min per recording. The counts per group (6 to 13) agree. The neuron counts run to 198 (two WT
P9-11 recordings have 164 and 198), so the stated range is wrong. Most recordings have 2160 frames,
which is 9.2 min at 3.9 Hz, longer than the stated 7 to 8 min; the range is printed above. The
descriptor gives no unit for the estimated rates. They are 0 in about 90% of frames, with
non-zero values from about 0.5 upwards, and the mean per recording is 0.19 to 0.74, close to the
Hz the paper reports (0.30 Hz in WT at P14-16, Fig. 6); per frame they would be 3.9 times that
(about 1.2 Hz). The values do not settle the unit (cell above): they are continuous (about 1.06 million distinct
values in 1.18 million), and they fall near multiples of 1 or of 3.9 no more often than continuous
values would, as deconvolved estimates need not be whole spikes. So U-a rests only on the means
lying near the paper's Hz values, which are among the values tested, and that argument is partly
circular. They are read as Hz below (reading U-a), and the claims on rates and on peaks of
synchrony also show U-b.
The distance matrices are symmetric with zero diagonals, and match the neuron counts.

In [ ]:
folder_group = {v: k for k, v in ssc8.TIFF_FOLDER.items()}
mv = pd.DataFrame([{k: v for k, v in m.items() if k != "frame_mean"} for m in movies])
tifs = mv[mv.tiff].copy()
print("Archive members:", len(mv), "; TIFF movies:", len(tifs), "; other members (not TIFF):")
print(mv[~mv.tiff][["path", "bytes", "error"]].to_string(index=False))
tifs["genotype"] = [folder_group[f][0] for f in tifs.folder]
tifs["age"] = [ssc8.AGE_LABEL[folder_group[f][1]] for f in tifs.folder]
tifs["index"] = [int(n.split("_")[1].split(".")[0]) - 1 for n in tifs.name]
print("\nMovies per folder:")
print(tifs.groupby("folder").size().to_string())
merged = tifs.merge(info[["genotype", "age", "index", "frames", "neurons"]], on=["genotype", "age", "index"],
                    how="outer", suffixes=("_tiff", "_mat"), indicator=True)
print("\nMovies matched to a MAT recording by group and number (Movie_k is entry k):",
      merged._merge.value_counts().to_dict())
print("Frame count of the movie equals the rows of the rate matrix:",
      int((merged.frames_tiff == merged.frames_mat).sum()), "of", len(merged))
print("Frame shapes (height x width):", tifs.groupby(["height", "width"]).size().to_dict(), "; bits:", tifs.bits.unique().tolist())
dup = tifs[tifs.duplicated("md5", keep=False)]
print("Movies with identical bytes:", len(dup))

# Recordings deposited twice in the MAT file: identical matrices or identical cell traces.
col_hash = {}
for (g, a), recs in data.items():
    for i, r in enumerate(recs):
        for j in range(r["rates"].shape[1]):
            col = r["rates"][:, j]
            if col.any():
                col_hash.setdefault(col.tobytes(), set()).add((g, a, i))
shared = [v for v in col_hash.values() if len(v) > 1]
print("Non-silent cell traces found in more than one recording:", len(shared))

In [ ]:
# Do the rates share the movies' clock? Per recording, the correlation of the population rate (mean
# over cells) with the movie's mean frame brightness, high-passed by subtracting a 41-frame running
# median (a standard choice, to remove bleaching and drift), at lags of -5 to +5 frames; against
# every other movie of the data set at lag 0 (the wrong movie); and against its own movie with the
# rate shifted by 50 to 400 frames (shifted labels).
from scipy.ndimage import median_filter


def highpass(x):
    return x - median_filter(x, 41, mode="nearest")


folder_group = {v: k for k, v in ssc8.TIFF_FOLDER.items()}
raw = {(folder_group[m["folder"]], int(m["name"].split("_")[1].split(".")[0]) - 1): m["frame_mean"]
       for m in movies if m["tiff"]}
bright = {k: highpass(v) for k, v in raw.items()}
lags = np.arange(-5, 6)
clock = []
for (g, a), recs in data.items():
    for i, r in enumerate(recs):
        fm = bright[((g, a), i)]
        pop = r["rates"].mean(1)
        lc = [np.corrcoef(np.roll(pop, l)[5:-5], fm[5:-5])[0, 1] for l in lags]
        others = []
        for key, f2 in bright.items():
            if key != ((g, a), i):
                n = min(len(pop), len(f2))
                others.append(np.corrcoef(pop[:n], f2[:n])[0, 1])
        shifted = [np.corrcoef(np.roll(pop, s), fm)[0, 1] for s in range(50, 401, 50)]
        # concatenation boundaries every 720 frames: brightness step there against elsewhere
        step = np.abs(np.diff(raw[((g, a), i)]))
        bnd = [k - 1 for k in range(720, len(step) + 1, 720)]
        clock.append({"genotype": g, "age": ssc8.AGE_LABEL[a], "index": i, "r_lag0": lc[5],
                      "best_lag": int(lags[np.argmax(lc)]), "r_best": max(lc),
                      "max_r_other_movie": max(others), "max_abs_r_shifted": max(np.abs(shifted)),
                      "step_at_720k_over_median": np.median(step[bnd]) / np.median(step) if bnd else np.nan})
clock = pd.DataFrame(clock)
print(clock.to_string(index=False))
print("\nOwn movie, lag 0: r = %.2f to %.2f (median %.2f); other movies: up to %.2f; own movie with shifted rates: up to %.2f"
      % (clock.r_lag0.min(), clock.r_lag0.max(), clock.r_lag0.median(), clock.max_r_other_movie.max(), clock.max_abs_r_shifted.max()))
print("Recordings whose own movie beats every one of the 52 other movies at lag 0:",
      int((clock.r_lag0 > clock.max_r_other_movie).sum()), "of", len(clock))
print("Recordings whose lag-0 correlation beats every shifted one:", int((clock.r_lag0 > clock.max_abs_r_shifted).sum()), "of", len(clock))
print("Lag (frames, + = brightness after the rate) of the best correlation:", clock.best_lag.value_counts().sort_index().to_dict())
print("Median ratio of the brightness step at frames 720k to the median step: %.1f" % clock.step_at_720k_over_median.median())

The movies are in one folder per group (the folders are named "4Hz", where the Methods and the
descriptor give 3.9 Hz). Each movie `Movie_k.tif` has exactly as many frames as the k-th rate
matrix of its group. The population rate follows the movie's high-passed brightness, best when the
brightness lags the rate by one or two frames, as a calcium signal lags the spikes it is
deconvolved from. Every rate matrix correlates better with its own movie at lag 0 than with any
of the 52 other movies, and than with its own movie when the rates are shifted by 50 to 400 frames
(counts above). So the rate matrices and the movies share one frame clock, and the k-th entry of
each group is `Movie_k.tif`; this is shown for every recording, though the margin is small for
some (KO P30-40 entry 0). Frames are 128 x 256 pixels in 50 movies (one each of 357, 404 and 512
wide), where the Methods give 512 x 128 at 3.9 Hz. Most recordings are 2160 frames, three times
720, which at 4 Hz is three 3-min movies, consistent with the Methods' "several 3-min calcium
movies were concatenated" and with the folder names; the brightness steps at multiples of 720
frames, larger than elsewhere, fit this. The TIFFs carry no time stamps, so the absolute frame rate
cannot be checked; 3.9 Hz (Methods) is used, with 4 Hz as a variant where a rate per second is
computed. No movie or cell trace is deposited twice.

In [ ]:
MEASURES = ["xcorr0", "spearman", "pearson"]
rows = []
for meas in MEASURES:
    for (g, a), recs in data.items():
        for i, r in enumerate(recs):
            rows.append({"measure": meas, "genotype": g, "age": a, "index": i, **ssc8.mouse_pair_means(r, meas)})
corr = pd.DataFrame(rows)
print("Pairs left out because a cell is silent (correlation undefined):",
      corr[corr.measure == "xcorr0"].n_undefined.sum(), "in",
      int((corr[corr.measure == "xcorr0"].n_undefined > 0).sum()), "recording(s)")
summary = (corr.groupby(["measure", "genotype", "age"], sort=False)
           .agg(n=("all", "size"), lt100=("<100", "mean"), lt100_sem=("<100", ssc8.sem),
                b100_200=("100-200", "mean"), gt200=(">200", "mean"), all_pairs=("all", "mean"),
                all_sem=("all", ssc8.sem), pairs_lt100=("n_<100", "sum")))
print(summary.to_string())
print("Range of mouse values, pairs under 100 um, xcorr0: %.3f to %.3f"
      % (corr[corr.measure == "xcorr0"]["<100"].min(), corr[corr.measure == "xcorr0"]["<100"].max()))


def values(meas, col="<100"):
    d = corr[corr.measure == meas]
    return {(g, a): d[(d.genotype == g) & (d.age == a)][col].to_numpy() for g, a in ssc8.GROUPS}


def within(x, value, tol):
    return abs(x - value) <= tol + 1e-12

## group-sizes

> The calcium imaging of unanesthetized mice comprises 12, 9 and 8 WT mice and 9, 10 and 8 Fmr1 -/-
> mice at P9-11, P14-16 and P30-40.

In [ ]:
expected = {("WT", "P9P11"): 12, ("WT", "P14P16"): 9, ("WT", "P30P40"): 8,
            ("KO", "P9P11"): 9, ("KO", "P14P16"): 10, ("KO", "P30P40"): 8}
gs = pd.DataFrame([{"genotype": g, "age": ssc8.AGE_LABEL[a], "mat_recordings": len(data[(g, a)]),
                    "tiff_movies": int(((tifs.genotype == g) & (tifs.age == ssc8.AGE_LABEL[a])).sum()),
                    "paper_fig1e": expected[(g, a)]} for g, a in ssc8.GROUPS])
gs["match"] = gs.mat_recordings == gs.paper_fig1e
print(gs.to_string(index=False))
print("Totals: WT %d (paper 29), KO %d (paper 27)" % (gs[gs.genotype == "WT"].mat_recordings.sum(), gs[gs.genotype == "KO"].mat_recordings.sum()))
print("P14-16 against the n of the peaks of synchrony sentence (9 KO, 10 WT): KO %d, WT %d"
      % (len(data[("KO", "P14P16")]), len(data[("WT", "P14P16")])))
print("Groups matching Fig. 1e:", int(gs.match.sum()), "of 6")

**Result: not reproduced.** Two of the six groups match Fig. 1e (KO P9-11, 9, and KO P14-16, 10). The
data set holds 13, 8 and 7 WT mice and 9, 10 and 6 KO mice, 28 WT and 25 KO in all, against 29 and
27 in the paper. The P14-16 counts (8 WT, 10 KO) match neither the Fig. 1e counts (9 WT, 10 KO) nor
those of the peaks of synchrony sentence (10 WT, 9 KO). The MAT file and the movies agree on every
count.

## fig1d-distance-dependent-decorrelation

> In WT mice, the drop in pairwise correlation from P9-11 (12 mice) to P14-16 (9 mice) is largest for
> cells less than 100 um apart (mean difference 0.223) and smaller for cells 100-200 um (0.159) and
> more than 200 um apart (0.067, not significant).

WT mice at P9-11 and P14-16 (13 and 8 in the data set). Readings of the ANOVA, from the criterion:
A-a, ordinary two-way ANOVA (age by bin, both between); post hoc per bin, the age difference
against the ANOVA's pooled residual mean square, Bonferroni over three bins. A-b, mouse as a
repeated measure over bins (split-plot ANOVA); post hoc per bin as an unpaired t-test within the
bin, Bonferroni over three bins. Some mice have no pairs over 200 um, or few; they are counted
below.

In [ ]:
EXP_1D = [0.223, 0.159, 0.067]
TOL_1D = [0.0223, 0.0159, 0.0067]
BIN_NAMES = ["<100", "100-200", ">200"]


def fig1d(meas):
    d = corr[(corr.measure == meas) & (corr.genotype == "WT") & corr.age.isin(["P9P11", "P14P16"])]
    long = d.melt(id_vars=["age", "index"], value_vars=BIN_NAMES, var_name="bin", value_name="r").dropna()
    long["mouse"] = long.age + "_" + long["index"].astype(str)
    # A-a: ordinary two-way ANOVA
    fit = ols("r ~ C(age) * C(bin)", data=long).fit()
    aov = sm.stats.anova_lm(fit, typ=2)
    mse, dfr = fit.mse_resid, fit.df_resid
    out = []
    for b in BIN_NAMES:
        x = long[(long.bin == b) & (long.age == "P9P11")].r.to_numpy()
        y = long[(long.bin == b) & (long.age == "P14P16")].r.to_numpy()
        diff = x.mean() - y.mean()
        t_a = diff / np.sqrt(mse * (1 / len(x) + 1 / len(y)))
        p_a = min(1.0, 3 * 2 * stats.t.sf(abs(t_a), dfr))
        p_b = min(1.0, 3 * stats.ttest_ind(x, y).pvalue)
        out.append({"bin": b, "n_P9-11": len(x), "n_P14-16": len(y), "mean_P9-11": x.mean(), "mean_P14-16": y.mean(),
                    "difference": diff, "p_bonf_A-a": p_a, "p_bonf_A-b": p_b})
    out = pd.DataFrame(out)
    # A-b: split-plot ANOVA on mice with all three bins
    full = long.groupby("mouse").filter(lambda g: len(g) == 3)
    return out, aov, full


def split_plot(full):
    # between factor age, within factor bin, subject mouse
    gm = full.r.mean()
    a_levels, b_levels = full.age.unique(), BIN_NAMES
    ss_age = sum(len(g) * (g.r.mean() - gm) ** 2 for _, g in full.groupby("age"))
    ss_subj = sum(3 * (g.r.mean() - gm) ** 2 for _, g in full.groupby("mouse"))
    ss_bin = sum(len(g) * (g.r.mean() - gm) ** 2 for _, g in full.groupby("bin"))
    ss_cells = sum(len(g) * (g.r.mean() - gm) ** 2 for _, g in full.groupby(["age", "bin"]))
    ss_ab = ss_cells - ss_age - ss_bin
    ss_tot = ((full.r - gm) ** 2).sum()
    n_subj = full.mouse.nunique()
    df_age, df_sw = len(a_levels) - 1, n_subj - len(a_levels)
    ss_sw = ss_subj - ss_age
    ss_err = ss_tot - ss_subj - ss_bin - ss_ab
    df_bin, df_ab, df_err = 2, 2 * df_age, 2 * df_sw
    f_age = (ss_age / df_age) / (ss_sw / df_sw)
    f_bin = (ss_bin / df_bin) / (ss_err / df_err)
    f_ab = (ss_ab / df_ab) / (ss_err / df_err)
    return pd.DataFrame({"F": [f_age, f_bin, f_ab], "df1": [df_age, df_bin, df_ab], "df2": [df_sw, df_err, df_err],
                         "p": [stats.f.sf(f_age, df_age, df_sw), stats.f.sf(f_bin, df_bin, df_err), stats.f.sf(f_ab, df_ab, df_err)]},
                        index=["age", "bin", "age x bin"])


def judge_1d(out, col):
    within_tol = all(within(d, e, t) for d, e, t in zip(out.difference, EXP_1D, TOL_1D))
    p_ok = out[col].iloc[0] < 0.05 and out[col].iloc[1] < 0.05 and out[col].iloc[2] >= 0.05
    falls = out.difference.iloc[0] > out.difference.iloc[1] > out.difference.iloc[2]
    return within_tol, p_ok, falls


for meas in ["xcorr0", "spearman"]:
    out, aov, full = fig1d(meas)
    print(f"=== measure {meas}" + (" (variant only)" if meas == "spearman" else " (reading C-b)"))
    print(out.to_string(index=False))
    print("expected differences", EXP_1D, "+/-", TOL_1D)
    print("A-a two-way ANOVA:\n", aov.to_string())
    print(f"A-b split-plot ANOVA on {full.mouse.nunique()} mice with pairs in all three bins:\n", split_plot(full).to_string())
    for col in ["p_bonf_A-a", "p_bonf_A-b"]:
        w, p, f = judge_1d(out, col)
        print(f"{col}: differences within tolerance {w}; first two p < 0.05 and third not {p}; falls with distance {f}")

In [ ]:
# Bin edges are fixed by the text; the share of mice without pairs over 200 um, and how the per-bin
# difference moves when the mice lacking a bin are dropped from every bin.
d = corr[(corr.measure == "xcorr0") & (corr.genotype == "WT") & corr.age.isin(["P9P11", "P14P16"])]
print(d[["age", "index", "n_<100", "n_100-200", "n_>200", "<100", "100-200", ">200"]].to_string(index=False))
full = d.dropna(subset=BIN_NAMES)
print("Mice with pairs in all bins: P9-11 %d of %d, P14-16 %d of %d" % (
    (full.age == "P9P11").sum(), (d.age == "P9P11").sum(), (full.age == "P14P16").sum(), (d.age == "P14P16").sum()))
for b in BIN_NAMES:
    x, y = full[full.age == "P9P11"][b], full[full.age == "P14P16"][b]
    print(b, "difference on those mice: %.4f; t-test within the bin, Bonferroni p = %.3g"
          % (x.mean() - y.mean(), min(1.0, 3 * stats.ttest_ind(x, y).pvalue)))
p14_gt200 = d[(d.age == "P14P16") & d[">200"].notna()]
print("P14-16 mice with pairs over 200 um:", len(p14_gt200), "; their pair counts:", p14_gt200["n_>200"].tolist())

**Result: partially reproduced.** In WT mice the drop in correlation from P9-11 (13 mice) to P14-16
(8 mice) is largest for pairs under 100 um and falls with distance: 0.191, 0.105 and 0.025 under
the main reading (C-b), with Bonferroni p below 0.05 for the first two bins and 1.0 for the third,
under both ANOVA readings. The pattern and the significance of the paper hold, but all three
differences are below the paper's (0.223, 0.159, 0.067) and outside the 10% tolerance; the
Spearman variant gives 0.175, 0.108 and 0.051, also outside. Only 4 of the 8 P14-16 mice have pairs
over 200 um (28, 108, 72 and 1 pairs), and one P9-11 mouse has none, so the result of the third bin
rests partly on four mice. On the 16 mice with all three bins, the split-plot sample, the
differences are 0.188, 0.102 and 0.025 and the within-bin t-tests give Bonferroni p = 0.0012,
0.022 and 1, the same pattern.

## fig1e-age-decrease

> For cell pairs within 100 um, correlation decreases significantly with age in both WT and Fmr1 -/-
> mice.

Readings from the criterion: E-a, the 2-df age contrast within each genotype of the unequal-variance
2 x 3 model; E-b, Welch's one-way ANOVA of age per genotype.

In [ ]:
def age_tests(meas):
    v = values(meas)
    m = ssc8.two_by_three(v)
    rows = []
    for g in ssc8.GENOTYPES:
        wa = ssc8.welch_anova([v[(g, a)] for a in ssc8.AGES])
        rows.append({"genotype": g, **{f"mean_{ssc8.AGE_LABEL[a]}": v[(g, a)].mean() for a in ssc8.AGES},
                     "E-a F": m[f"age_within_{g}"]["F"], "E-a df2": m[f"age_within_{g}"]["df2"], "E-a p": m[f"age_within_{g}"]["p"],
                     "E-b F": wa["F"], "E-b df2": wa["df2"], "E-b p": wa["p"],
                     "P9-11 > P30-40": v[(g, "P9P11")].mean() > v[(g, "P30P40")].mean()})
    return pd.DataFrame(rows)


for meas in ["xcorr0", "spearman"]:
    print(f"=== measure {meas}" + (" (variant only)" if meas == "spearman" else " (reading C-b)"))
    print(age_tests(meas).to_string(index=False))

**Result: reproduced.** For pairs under 100 um correlation falls with age in both genotypes, from
0.360 to 0.169 to 0.119 in WT and from 0.421 to 0.204 to 0.134 in KO (C-b). The age contrast of the
unequal-variance model gives F = 52.5 (WT) and 64.3 (KO), p far below 0.01 in both; Welch's one-way
ANOVA gives the same values, as the two tests coincide for a one-way design. The Spearman variant
agrees.

## genotype-pooled-ages

> Pooling all ages, correlation coefficients are higher in Fmr1 -/- than WT mice (0.265 +/- 0.009
> against 0.230 +/- 0.008, p = 0.0057).

Readings from the criterion: pairs P-a (under 100 um) and P-b (all pairs); pooled average M-a (mean
over all mice of a genotype, s.e.m. over mice) and M-b (mean of the three age means, s.e.m. from the
three age s.e.m.s). The test is the genotype main effect of the unequal-variance 2 x 3 model,
which is the same under M-a and M-b.

In [ ]:
def pooled(meas, exp, tol):
    rows = []
    for pairs, col in [("P-a (<100 um)", "<100"), ("P-b (all pairs)", "all")]:
        v = values(meas, col)
        test = ssc8.two_by_three(v)["genotype"]
        for avg in ["M-a", "M-b"]:
            r = {"pairs": pairs, "average": avg}
            for g in ssc8.GENOTYPES:
                if avg == "M-a":
                    x = np.concatenate([v[(g, a)] for a in ssc8.AGES])
                    r[f"{g}_mean"], r[f"{g}_sem"] = x.mean(), ssc8.sem(x)
                else:
                    r[f"{g}_mean"] = np.mean([v[(g, a)].mean() for a in ssc8.AGES])
                    r[f"{g}_sem"] = np.sqrt(sum(ssc8.sem(v[(g, a)]) ** 2 for a in ssc8.AGES)) / 3
            r["F"], r["df2"], r["p"] = test["F"], test["df2"], test["p"]
            r["KO higher"] = r["KO_mean"] > r["WT_mean"]
            r["WT within"] = within(r["WT_mean"], exp["WT"], tol["WT"])
            r["KO within"] = within(r["KO_mean"], exp["KO"], tol["KO"])
            rows.append(r)
    return pd.DataFrame(rows)


EXP_POOL, TOL_POOL = {"WT": 0.230, "KO": 0.265}, {"WT": 0.008, "KO": 0.009}
for meas in ["xcorr0", "spearman"]:
    print(f"=== measure {meas}" + (" (variant only)" if meas == "spearman" else " (reading C-b)"))
    print(pooled(meas, EXP_POOL, TOL_POOL).to_string(index=False))
print("expected WT 0.230 +/- 0.008, KO 0.265 +/- 0.009")

**Result: partially reproduced.** The genotype main effect of the unequal-variance 2 x 3 model is
significant with KO higher: p = 0.0064 for pairs under 100 um (paper 0.0057) and 0.0028 for all
pairs. No reading puts both pooled means within the paper's s.e.m.: pairs under 100 um averaged
over mice give WT 0.245 and KO 0.265 (KO within, WT not); all pairs over mice give WT 0.223 and KO
0.251 (WT within, KO not); the mean of the age means gives WT 0.216 and KO 0.253 (under 100 um),
whose s.e.m.s (0.0088 and 0.0093) are close to the paper's (0.008 and 0.009), while those over mice
(0.024 and 0.026) are not. This points to the mean of the age means (M-b) as the paper's form, and
under it both means are about 0.013 below the paper's, outside tolerance.

## pearson-genotype

> With Pearson correlation, pooled over ages, coefficients are again higher in Fmr1 -/- than WT mice
> (0.196 +/- 0.010 against 0.164 +/- 0.009, p = 0.026).

As genotype-pooled-ages, with the zero-lag Pearson correlation.

In [ ]:
EXP_PEAR, TOL_PEAR = {"WT": 0.164, "KO": 0.196}, {"WT": 0.009, "KO": 0.010}
print(pooled("pearson", EXP_PEAR, TOL_PEAR).to_string(index=False))
print("expected WT 0.164 +/- 0.009, KO 0.196 +/- 0.010")
v = values("pearson")
print("Pearson, pairs under 100 um, per group:", {f"{g} {ssc8.AGE_LABEL[a]}": round(v[(g, a)].mean(), 3) for g, a in ssc8.GROUPS})

**Result: partially reproduced.** With Pearson correlation the genotype effect is significant with
KO higher: p = 0.023 for pairs under 100 um (paper 0.026) and 0.012 for all pairs. No reading puts
both means within the paper's s.e.m.: under 100 um over mice WT 0.179 and KO 0.193 (KO within);
all pairs over mice WT 0.155 and KO 0.178 (WT within); the means of the age means are 0.146 and
0.180 (under 100 um), with s.e.m.s 0.0092 and 0.0107, close to the paper's 0.009 and 0.010, which points to M-b as the
paper's form; under it both means are below the paper's (by 0.018 and 0.016), outside tolerance.

## fig1ef-p14-16-difference

> The genotype difference is largest at P14-16, where correlation is 0.210 +/- 0.011 in Fmr1 -/-
> against 0.172 +/- 0.012 in WT mice (p = 0.032).

Pairs under 100 um, mouse means, unpaired two-tailed Student's t-test (Online Methods, Statistical
Analyses). The KO minus WT difference is printed for every age.

In [ ]:
def per_age(meas, col="<100"):
    v = values(meas, col)
    rows = []
    for a in ssc8.AGES:
        w, k = v[("WT", a)], v[("KO", a)]
        rows.append({"age": ssc8.AGE_LABEL[a], "n_WT": len(w), "WT": w.mean(), "WT_sem": ssc8.sem(w),
                     "n_KO": len(k), "KO": k.mean(), "KO_sem": ssc8.sem(k), "KO-WT": k.mean() - w.mean(),
                     "t": stats.ttest_ind(k, w).statistic, "p_student": stats.ttest_ind(k, w).pvalue,
                     "p_welch": stats.ttest_ind(k, w, equal_var=False).pvalue})
    return pd.DataFrame(rows)


for meas in ["xcorr0", "spearman"]:
    print(f"=== measure {meas}" + (" (variant only)" if meas == "spearman" else " (reading C-b)"))
    t = per_age(meas)
    print(t.to_string(index=False))
    p14 = t[t.age == "P14-16"].iloc[0]
    print("P14-16 WT within 0.172 +/- 0.012:", within(p14.WT, 0.172, 0.012), "; KO within 0.210 +/- 0.011:", within(p14.KO, 0.210, 0.011),
          "; p < 0.05:", p14.p_student < 0.05, "; KO higher:", p14.KO > p14.WT,
          "; P14-16 difference largest:", bool(p14["KO-WT"] > t[t.age != "P14-16"]["KO-WT"].max()))
    print("Variant, KO / WT ratio per age:", dict(zip(t.age, (t.KO / t.WT).round(3))))

In [ ]:
# Per-mouse values at P14-16, the check of variances the Methods describe ("variances ... to ensure
# that a suitable statistical test was chosen"), and the Student p with each mouse left out in turn.
v14 = values("xcorr0")
w14, k14 = v14[("WT", "P14P16")], v14[("KO", "P14P16")]
print("P14-16 mouse values, xcorr0, pairs under 100 um")
print("  WT:", np.round(w14, 3).tolist())
print("  KO:", np.round(k14, 3).tolist())
print("s.d. WT %.4f, KO %.4f; variance ratio KO / WT %.2f" % (w14.std(ddof=1), k14.std(ddof=1), k14.var(ddof=1) / w14.var(ddof=1)))
print("Levene (median) p = %.3f; Levene (mean) p = %.3f; Bartlett p = %.3f; two-sided F-test p = %.3f" % (
    stats.levene(k14, w14).pvalue, stats.levene(k14, w14, center="mean").pvalue, stats.bartlett(k14, w14).pvalue,
    min(1.0, 2 * min(stats.f.sf(k14.var(ddof=1) / w14.var(ddof=1), len(k14) - 1, len(w14) - 1),
                     stats.f.cdf(k14.var(ddof=1) / w14.var(ddof=1), len(k14) - 1, len(w14) - 1)))))
for g, x in [("WT", w14), ("KO", k14)]:
    loo = [stats.ttest_ind(k14 if g == "WT" else np.delete(k14, j), np.delete(w14, j) if g == "WT" else w14).pvalue
           for j in range(len(x))]
    print(f"{g} mouse left out, Student p:", np.round(loo, 3).tolist(),
          f"; below 0.05 in {sum(q < 0.05 for q in loo)} of {len(loo)}")

**Result: not reproduced.** At P14-16 the means match the paper (WT 0.169 +/- 0.010, 8 mice; KO 0.204
+/- 0.013, 10 mice; tolerances 0.172 +/- 0.012 and 0.210 +/- 0.011), and KO is higher, but the
unpaired Student's t-test gives p = 0.057, above 0.05 (Welch's test, not the Methods' test, gives
0.049). The KO minus WT difference is 0.062 at P9-11, 0.035 at P14-16 and 0.015 at P30-40, so it is
not largest at P14-16; as a ratio (a variant the criterion does not use) it is largest there (1.21,
against 1.17 and 1.13). The Spearman variant gives p = 0.058 at P14-16. The variances do not differ (s.d. 0.029 and 0.040;
Levene p = 0.31, Bartlett p = 0.39, F-test p = 0.40), so the Methods' check of variances leads to
Student's test. The result is fragile: with one mouse left out, p runs from 0.018 to 0.109, below
0.05 in 1 of 8 WT and 3 of 10 KO cases, and dropping the WT mouse of highest correlation (0.228)
gives p = 0.018. As the data set has one WT mouse fewer at P14-16 than the paper, the sample
difference alone could move the test across 0.05.

## p30-40-no-difference

> By P30-40 the genotype difference in correlation has decreased and is not significant (0.144 +/-
> 0.008 in Fmr1 -/- against 0.123 +/- 0.008 in WT, p = 0.075).

A claim of absence, judged on the unpaired two-tailed t-test at alpha 0.05, under reading C-b (the
reading of fig1ef-p14-16-difference, the only one here that can decide).

In [ ]:
for meas in ["xcorr0", "spearman"]:
    t = per_age(meas)
    p30 = t[t.age == "P30-40"].iloc[0]
    print(f"{meas}: WT {p30.WT:.4f} +/- {p30.WT_sem:.4f} (n {p30.n_WT}), KO {p30.KO:.4f} +/- {p30.KO_sem:.4f} (n {p30.n_KO}), "
          f"KO-WT {p30['KO-WT']:.4f}, Student p = {p30.p_student:.4f}, Welch p = {p30.p_welch:.4f}; "
          f"WT within 0.123 +/- 0.008: {within(p30.WT, 0.123, 0.008)}, KO within 0.144 +/- 0.008: {within(p30.KO, 0.144, 0.008)}")
d = corr[(corr.measure == "xcorr0") & (corr.age == "P30P40")]
print("\nP30-40 mouse values, xcorr0, pairs under 100 um:")
print(d[["genotype", "index", "<100", "n_<100"]].to_string(index=False))

**Result: reproduced.** At P30-40 the t-test finds no genotype difference: KO 0.134 +/- 0.011 (6 mice)
against WT 0.119 +/- 0.005 (7 mice), difference 0.015, p = 0.22. WT is within the paper's s.e.m.
(0.123 +/- 0.008); KO is below it (0.144 +/- 0.008).

## fig2b-peak-frequency

> The frequency of peaks of synchrony at P14-16 is slightly but not significantly higher in Fmr1 -/-
> mice (0.060 +/- 0.007 against 0.053 +/- 0.006 peaks per s, p = 0.45).

Peaks of synchrony as the Online Methods define them, under the readings of the criterion: active
R-a (rate above 0) or R-b (at least one estimated spike in the frame); shuffle S-a (permutation of
a trace's frames) or S-b (circular shift). R-b depends on the rate unit: with the rates in Hz (U-a)
it is rate / 3.9 >= 1, labelled R-b; with the rates per frame (U-b) it is rate >= 1, labelled
R-b/U-b. Both shuffles keep each cell's mean rate. Frequency is events over the recording's
duration at 3.9 Hz (4 Hz as a variant). A claim of absence: unpaired two-tailed t-test at alpha
0.05 on the data set's P14-16 mice (8 WT, 10 KO). Each reading has its own seeded generator.


In [ ]:
# R-b/U-b is added last, so that the generators of the first four readings keep their seeds.
READINGS = [(r, s) for r in ["R-a", "R-b", "R-b/U-b"] for s in ["S-a", "S-b"]]
ev_rows = []
for k, (rule, how) in enumerate(READINGS):
    rng = np.random.default_rng(2013 + k)
    for g in ssc8.GENOTYPES:
        for i, r in enumerate(data[(g, "P14P16")]):
            e = ssc8.synchrony_events(r["rates"], rule, how, rng)
            ev_rows.append({"rule": rule, "shuffle": how, "genotype": g, "index": i, **e})
ev = pd.DataFrame(ev_rows)
print(ev[(ev["rule"] == "R-a") & (ev.shuffle == "S-a")][["genotype", "index", "frames", "threshold", "n_events", "rate_per_s", "Q-a", "Q-b"]].to_string(index=False))
rows = []
for (rule, how), d in ev.groupby(["rule", "shuffle"]):
    w, k = d[d.genotype == "WT"], d[d.genotype == "KO"]
    for col in ["rate_per_s", "rate_per_s_4hz"]:
        rows.append({"rule": rule, "shuffle": how, "measure": col, "WT": w[col].mean(), "WT_sem": ssc8.sem(w[col]),
                     "KO": k[col].mean(), "KO_sem": ssc8.sem(k[col]), "p_student": stats.ttest_ind(k[col], w[col]).pvalue,
                     "WT within": within(w[col].mean(), 0.053, 0.006), "KO within": within(k[col].mean(), 0.060, 0.007),
                     "threshold_range": f"{d.threshold.min():.1f}-{d.threshold.max():.1f}%", "mice_without_events": int((d.n_events == 0).sum())})
fb = pd.DataFrame(rows)
print("\nexpected KO 0.060 +/- 0.007, WT 0.053 +/- 0.006 peaks per s")
print(fb.to_string(index=False))
print("Range of per-mouse frequencies, all readings: %.4f to %.4f per s" % (ev.rate_per_s.min(), ev.rate_per_s.max()))

**Result: reproduced.** At P14-16 the frequency of peaks of synchrony does not differ between
genotypes under any of the six readings (p = 0.13 to 0.96). Under R-a, the reading of "active" that
gives the paper's values, both means are within the paper's s.e.m. with either shuffle: S-a gives
KO 0.060 +/- 0.006 and WT 0.054 +/- 0.007 per s (p = 0.52), S-b KO 0.061 and WT 0.057 (p = 0.64).
Under R-b with the rates in Hz (rate / 3.9 >= 1) events are rarer (0.016 to 0.027 per s) and
outside the tolerances. Under R-b with the rates per frame (R-b/U-b, rate >= 1, which 97% of the
non-zero values meet) the frequencies are close to R-a: KO 0.055 and WT 0.053 with S-a (p = 0.82),
KO 0.059 and WT 0.059 with S-b (p = 0.96). A 4 Hz frame rate
changes the frequencies by 2.6% and not the tests.

## fig2c-participation

> At P14-16 a higher proportion of neurons participates in peaks of synchrony in Fmr1 -/- than in WT
> mice (0.380 +/- 0.023 against 0.300 +/- 0.017, p = 0.023).

Events as in fig2b-peak-frequency, under its readings R and S; proportion Q-a (per event, the fraction
of the mouse's cells active in any frame of the event, averaged over events) or Q-b (the fraction of
the mouse's cells active in at least one event). Unpaired two-tailed t-test. Mice without events
have no Q-a and are dropped, and counted.

In [ ]:
rows = []
for (rule, how), d in ev.groupby(["rule", "shuffle"]):
    for q in ["Q-a", "Q-b"]:
        w, k = d[d.genotype == "WT"][q].dropna(), d[d.genotype == "KO"][q].dropna()
        rows.append({"rule": rule, "shuffle": how, "measure": q, "n_WT": len(w), "WT": w.mean(), "WT_sem": ssc8.sem(w),
                     "n_KO": len(k), "KO": k.mean(), "KO_sem": ssc8.sem(k), "p_student": stats.ttest_ind(k, w).pvalue,
                     "KO higher": k.mean() > w.mean(), "WT within": within(w.mean(), 0.300, 0.017),
                     "KO within": within(k.mean(), 0.380, 0.023)})
fc = pd.DataFrame(rows)
print("expected KO 0.380 +/- 0.023, WT 0.300 +/- 0.017")
print(fc.to_string(index=False))
# How the per-event participation depends on the event size: the mean percent active in all frames.
print("\nMean percent of cells active per frame, R-a: WT %.1f%%, KO %.1f%%" % (
    ev[(ev["rule"] == "R-a") & (ev.genotype == "WT")].mean_pct_active.mean(), ev[(ev["rule"] == "R-a") & (ev.genotype == "KO")].mean_pct_active.mean()))

In [ ]:
# Seed sensitivity of the closest reading, R-b/U-b with Q-a. The threshold is the 95th percentile
# of 100 shuffled peaks (Online Methods), so it varies with the generator's seed. Twenty further
# seeds per shuffle, then the limit the threshold tends to with 1000 shuffles (S-a, whose
# threshold settles on a value of the discrete percent active, with one seed; S-b with three).
def fig2c_ub(how, seed, n_shuffles=ssc8.N_SHUFFLES):
    rng = np.random.default_rng(seed)
    q = {g: np.array([ssc8.synchrony_events(r["rates"], "R-b/U-b", how, rng, n_shuffles)["Q-a"]
                      for r in data[(g, "P14P16")]]) for g in ssc8.GENOTYPES}
    w, k = q["WT"][~np.isnan(q["WT"])], q["KO"][~np.isnan(q["KO"])]
    return {"shuffle": how, "seed": seed, "n_shuffles": n_shuffles, "WT": w.mean(), "KO": k.mean(),
            "KO higher": k.mean() > w.mean(), "p_student": stats.ttest_ind(k, w).pvalue}

seeds = pd.DataFrame([fig2c_ub(how, 5000 + s) for how in ["S-a", "S-b"] for s in range(20)])
print(seeds.to_string(index=False))
for how, d in seeds.groupby("shuffle"):
    print(f"{how}, 20 seeds of 100 shuffles: p median {d.p_student.median():.3f}, range {d.p_student.min():.3f} "
          f"to {d.p_student.max():.3f}, p < 0.05 with KO higher in {int(((d.p_student < 0.05) & d['KO higher']).sum())} of 20; "
          f"WT {d.WT.min():.3f}-{d.WT.max():.3f}, KO {d.KO.min():.3f}-{d.KO.max():.3f}")
big = pd.DataFrame([fig2c_ub("S-a", 6000, 1000)] + [fig2c_ub("S-b", 6000 + s, 1000) for s in range(3)])
print("\n1000 shuffles per threshold:")
print(big.to_string(index=False))


**Result: not reproduced.** No reading gives a significant genotype difference in the proportion of
neurons in peaks of synchrony (p = 0.052 to 0.76 over the twelve readings). Under R-a and Q-a KO is higher (0.444 against
0.411 with S-a, p = 0.25), but both are above the paper's 0.380 and 0.300 and outside their s.e.m.
Under Q-b with R-a nearly every cell takes part in some event (0.998 to 1.0). The closest reading is
R-b/U-b with Q-a (rates per frame, at least one spike): KO 0.437 against WT 0.391 with S-a (p =
0.052) and 0.429 against 0.385 with S-b (p = 0.053), KO higher but not significant, and both above
the paper's values and outside their s.e.m. That p sits at 0.05 and the threshold comes from 100
seeded shuffles, so the reading was run with 20 further seeds per shuffle. With S-a p runs from
0.056 to 0.116 (median 0.069) and is below 0.05 in none of 20; with S-b it runs from 0.044 to 0.116
(median 0.060) and is below 0.05, KO higher, in 3 of 20. With 1000 shuffles per threshold, which
removes most of the threshold's sampling noise, p is 0.056 (S-a) and 0.051 to 0.054 (S-b, three
seeds). The typical draw of the Methods' procedure, and its limit, do not reach p < 0.05, so the
outcome stays not reproduced; it is close to the boundary, and in a few draws the test holds. In
every draw both means are outside tolerance (KO 0.426 to 0.436 against 0.380 +/- 0.023), so
where the test holds the outcome would be partially reproduced.

## imaging-firing-rate-no-genotype-difference

> Firing rates extrapolated from the calcium imaging by deconvolution do not differ between genotypes
> at any age.

Per mouse, the mean over cells of each cell's mean estimated rate; unpaired two-tailed t-test per age
at alpha 0.05 (criterion). Reading U-a takes the rates as Hz; U-b as per frame (times 3.9), which
scales both groups alike and leaves the tests unchanged.

In [ ]:
fr = pd.DataFrame([{"genotype": g, "age": a, "index": i, "rate_Hz": r["rates"].mean(0).mean()}
                   for g, a in ssc8.GROUPS for i, r in enumerate(data[(g, a)])])
rows = []
for a in ssc8.AGES:
    w, k = fr[(fr.genotype == "WT") & (fr.age == a)].rate_Hz, fr[(fr.genotype == "KO") & (fr.age == a)].rate_Hz
    rows.append({"age": ssc8.AGE_LABEL[a], "n_WT": len(w), "WT_U-a": w.mean(), "WT_sem": ssc8.sem(w), "n_KO": len(k),
                 "KO_U-a": k.mean(), "KO_sem": ssc8.sem(k), "KO-WT": k.mean() - w.mean(),
                 "WT_U-b": 3.9 * w.mean(), "KO_U-b": 3.9 * k.mean(),
                 "p_student": stats.ttest_ind(k, w).pvalue, "p_welch": stats.ttest_ind(k, w, equal_var=False).pvalue})
rates_by_age = pd.DataFrame(rows)
print(rates_by_age.to_string(index=False))
print("Range of mouse rates: %.3f to %.3f Hz (U-a)" % (fr.rate_Hz.min(), fr.rate_Hz.max()))
print("Ages with p < 0.05:", rates_by_age[rates_by_age.p_student < 0.05].age.tolist())

**Result: reproduced.** Firing rates do not differ between genotypes at any age: KO minus WT is 0.017
Hz at P9-11 (p = 0.73), 0.0002 Hz at P14-16 (p = 0.997) and 0.014 Hz at P30-40 (p = 0.76).

## unanesthetized-p14-16-values

> In unanesthetized P14-16 mice the extrapolated firing rate is 0.30 +/- 0.03 Hz in WT and 0.29 +/-
> 0.03 Hz in Fmr1 -/- mice, and the correlation of pairs within 100 um is 0.17 +/- 0.01 in WT mice.

The unanesthetized groups of Fig. 6 are read as the P14-16 mice (criterion).

In [ ]:
p14 = rates_by_age[rates_by_age.age == "P14-16"].iloc[0]
c14 = per_age("xcorr0")
c14 = c14[c14.age == "P14-16"].iloc[0]
s14 = per_age("spearman")
s14 = s14[s14.age == "P14-16"].iloc[0]
for unit, wt, ko in [("U-a (Hz)", p14["WT_U-a"], p14["KO_U-a"]), ("U-b (per frame x 3.9)", p14["WT_U-b"], p14["KO_U-b"])]:
    print(f"{unit}: WT {wt:.3f} (0.30 +/- 0.03: {within(wt, 0.30, 0.03)}), KO {ko:.3f} (0.29 +/- 0.03: {within(ko, 0.29, 0.03)})")
print(f"s.e.m. of rates (U-a): WT {p14.WT_sem:.3f}, KO {p14.KO_sem:.3f}; n WT {p14.n_WT}, KO {p14.n_KO}")
print(f"WT correlation, pairs under 100 um, xcorr0: {c14.WT:.4f} +/- {c14.WT_sem:.4f} (0.17 +/- 0.01: {within(c14.WT, 0.17, 0.01)}); "
      f"Spearman variant {s14.WT:.4f}")

**Result: reproduced.** With the rates read as Hz (U-a), the P14-16 firing rate is 0.304 +/- 0.027 Hz
in WT (8 mice) and 0.304 +/- 0.027 Hz in KO (10 mice), within 0.30 +/- 0.03 and 0.29 +/- 0.03, and
the WT correlation of pairs under 100 um is 0.169 +/- 0.010, within 0.17 +/- 0.01. Read as per frame
(U-b) the rates would be 1.19 Hz, far outside.

## Variant for suppl4a-firing-vs-correlation (blocked)

The claim uses the 3-min recordings of the EEG subset (13 WT and 10 KO recordings), which the data set
does not identify, and it holds no EEG. The outcome is blocked. As a variant only, the P14-16
recordings of exactly 2160 frames are cut into three 720-frame segments (one 3-min movie each, as
the data check suggests; recordings of other lengths are left out, and the brightness step at
frames 720 and 1440 is printed for each), and each segment's mean rate is regressed on its mean correlation of pairs under
100 um (xcorr0). Segments of one mouse are not independent, so the F-test p is optimistic.

In [ ]:
rows = []
used = {"WT": 0, "KO": 0}
for g in ssc8.GENOTYPES:
    for i, r in enumerate(data[(g, "P14P16")]):
        x = r["rates"]
        if x.shape[0] != 2160:
            continue
        used[g] += 1
        for s in range(3):
            seg = {"rates": x[720 * s:720 * (s + 1)], "dist": r["dist"]}
            m = ssc8.mouse_pair_means(seg, "xcorr0")
            rows.append({"genotype": g, "mouse": i, "segment": s, "rate": seg["rates"].mean(), "corr": m["<100"]})
seg = pd.DataFrame(rows).dropna()
n14 = {g: len(data[(g, "P14P16")]) for g in ssc8.GENOTYPES}
print("P14-16 recordings of 2160 frames used: WT %d of %d, KO %d of %d" % (used["WT"], n14["WT"], used["KO"], n14["KO"]))
print(clock[(clock.age == "P14-16")][["genotype", "index", "step_at_720k_over_median"]].merge(
    info[info.age == "P14-16"][["genotype", "index", "frames"]], on=["genotype", "index"]).to_string(index=False))
for g in ssc8.GENOTYPES:
    d = seg[seg.genotype == g]
    lr = stats.linregress(d.rate, d["corr"])
    print(f"{g}: {len(d)} segments from {d.mouse.nunique()} mice, r2 = {lr.rvalue ** 2:.3f}, slope {lr.slope:.3f}, p = {lr.pvalue:.4f}")

The variant is not the paper's sample and decides nothing. On the 2160-frame P14-16 recordings
(4 WT and 6 KO mice) the KO regression across 3-min segments is positive (r2 = 0.216, p = 0.052,
18 segments) and the WT one also positive and not significant (r2 = 0.116, p = 0.28, 12 segments).
In each recording used, the median of the brightness steps at frames 720 and 1440 is above the
median step (ratio 1.08 to 13.5), consistent with movie boundaries there; for KO entry 4 (1.08) it is
only just above.


## Summary

| Claim | Outcome |
|---|---|
| group-sizes | not reproduced (13, 8, 7 WT and 9, 10, 6 KO mice against 12, 9, 8 and 9, 10, 8) |
| fig1d-distance-dependent-decorrelation | partially reproduced (pattern and tests hold, differences smaller) |
| fig1e-age-decrease | reproduced |
| genotype-pooled-ages | partially reproduced (p = 0.0064, means not both within s.e.m.) |
| pearson-genotype | partially reproduced (p = 0.023, means not both within s.e.m.) |
| fig1ef-p14-16-difference | not reproduced (means match, Student p = 0.057) |
| p30-40-no-difference | reproduced |
| fig2b-peak-frequency | reproduced |
| fig2c-participation | not reproduced |
| imaging-firing-rate-no-genotype-difference | reproduced |
| unanesthetized-p14-16-values | reproduced |
| seven whole-cell claims | blocked: data missing |
| five EEG claims | blocked: data missing |
| suppl4a-firing-vs-correlation | blocked: metadata insufficient (the recordings of the EEG subset are not identified) |
| fast-imaging-control | blocked: data missing |
| three isoflurane claims | blocked: data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

**How the analysis differs from the paper's.** The data set holds 53 mice, not the paper's 56, in
groups of different sizes (two of six match), so every group statistic is on a different sample.
The trace is the deposited estimated rate; the paper's correlation may have used the filtered dF/F
(reading T-b), which cannot be rebuilt as no cell contours are deposited. The correlation measure
of the paper (its ref. 15) is not given; the zero-lag normalised cross-correlation is the claims
file's reading, and it matches the paper's P14-16 and P30-40 WT values. The way the 2 x 3
unequal-variance ANOVA was fitted is not given; the Welch-James form used here reduces to the
Welch test for one-df contrasts. For peaks of synchrony, what counts as an active cell and how the
traces were shuffled are not given; all readings were run, with "at least one spike" read under
both rate units (six in all), and the outcomes do not depend on them. For fig2c the closest reading
(R-b/U-b, Q-a) gives p = 0.051 to 0.056 with 1000 shuffles and p < 0.05 in 3 of 40 seeded draws
of 100 shuffles, so that outcome is near the boundary. The rate unit is not given,
and the values do not show it; Hz is assumed because the means match the paper's Hz.

**What was found about the data set.** The group sizes differ from the paper's. The descriptor
gives 40 to 149 neurons per animal (the data run to 198) and 7 to 8 min per recording (most are
9.2 min at 3.9 Hz; the range is 3.8 to 15.4 min), and no unit for the estimated rates (their
means fit Hz; the values themselves do not show the unit). The movie folders say 4 Hz where the paper and descriptor say 3.9 Hz, and frames are 128 x
256 pixels where the Methods say 512 x 128. The archive carries five macOS resource-fork files.
Movies and rate matrices share their frame clock, and the k-th matrix of each group is
`Movie_k.tif`, which the descriptor does not say. One WT P30-40 cell is silent throughout. No
cell contours or dF/F traces are deposited, nor the EEG, whole-cell, isoflurane or 15.6 Hz data.

In [ ]:
print(f"Notebook run time: {(time.time() - started) / 60:.1f} min")